## Conversion BDB Eliran Wong → JSON

Conversion du dictionnaire BDB (Brown-Driver-Briggs Hebrew Lexicon) depuis le
fichier CSV d'Eliran Wong (`BDB-EliranWong.csv`) vers un format JSON structuré.

### Source

- **Fichier** : `BDB-EliranWong.csv` (10 022 entrées, format TSV)
- **Colonnes** : `BDBid`, `StrongNumber`, `content` (HTML propriétaire)

### Structure JSON produite

Chaque entrée est un objet avec les champs suivants :

| Champ | Type | Description |
|---|---|---|
| `sort_key` | int | Numéro d'ordre BDB (`BDB4` → `4`) |
| `strong` | str/null | Numéro Strong (`H24`), `null` si absent |
| `m` | str/null | Forme principale hébreu avec voyelles et accents |
| `b` | str/null | Forme nue sans diacritiques (pour la recherche) |
| `l` | str/null | Translitération latine |
| `d` | str | Définition complète HTML (balises `<p>` uniquement) |

### Fonctions à coder

- **`strip_vowels(text)`** — supprime les voyelles (niqqud) d'une forme hébraïque
- **`transliterate(text)`** — convertit une forme hébraïque en translitération latine via `TRANSLIT`
- **`extract_headword(content_html)`** — extrait le lemme principal : premier `<bdbheb>` dans le premier `<p>`
- **`parse_row(row)`** — convertit une ligne CSV en entrée JSON complète


In [82]:
from bs4 import BeautifulSoup
import csv, sys, re, json


In [83]:
TRANSLIT = {
    # Consonnes
    '\u05D0': "'",
    '\u05D1': 'b',
    '\u05D2': 'g',
    '\u05D3': 'd',
    '\u05D4': 'h',
    '\u05D5': 'w',
    '\u05D6': 'z',
    '\u05D7': 'kh',
    '\u05D8': 't',
    '\u05D9': 'y',
    '\u05DB': 'k',
    '\u05DA': 'k',
    '\u05DC': 'l',
    '\u05DE': 'm',
    '\u05DD': 'm',
    '\u05E0': 'n',
    '\u05DF': 'n',
    '\u05E1': 's',
    '\u05E2': "'",
    '\u05E4': 'p',
    '\u05E3': 'p',
    '\u05E6': 'ts',
    '\u05E5': 'ts',
    '\u05E7': 'q',
    '\u05E8': 'r',
    '\u05E9': 'sh',
    '\u05EA': 't',
    # Voyelles (niqqud)
    '\u05B0': 'e', '\u05B1': 'e', '\u05B2': 'a', '\u05B3': 'o',
    '\u05B4': 'i', '\u05B5': 'e', '\u05B6': 'e', '\u05B7': 'a',
    '\u05B8': 'a', '\u05B9': 'o', '\u05BA': 'o', '\u05BB': 'u',
    '\u05BC': '',  # dagesh (ignoré)
    '\u05BE': '-', # maqqef (tiret)
    '\u05C1': '',  # shin dot
    '\u05C2': '',  # sin dot
}

In [84]:
def strip_vowels(text): #Supprime les voyelles de 'text'
    return re.sub(r'[\u05B0-\u05C7\u05F0-\u05F4\uFB1D-\uFB4E]', '', text)

def transliterate(text, drop_gutturals=False):
    text = text.replace('ש\u05C1', 'sh').replace('ש\u05C2', 's')
    result = ''
    for char in text:
        if drop_gutturals and char in ('\u05D0', '\u05E2'):
            result += ''
        else:
            result += TRANSLIT.get(char, char)
    return result

def extract_definition(content):
    # cette fonction retourne une liste de tous les <p> contenu dans une ligne du csv
    content_soup = BeautifulSoup(content, features='html.parser')
    return content_soup.find_all('p') # -> trouve le premier <p> de l'article
    
def extract_headwords(d): #Extrait les entrées hébreux du dictionnaire
    entry = d.find(['bdbheb', 'bdbarc'])  # -> cherche le premier mot hébreu ou araméen de l'article
    words = []
    while entry:
        words.append(entry.text)
        nxt = entry.next_sibling # -> cherche l'entrée suivante
        if nxt is None or not str(nxt).strip().startswith(','): # -> si l'entrée suivante n'est pas une virgule
            break
        else:
            candidate = nxt.next_sibling
            if candidate and candidate.name in ['bdbheb', 'bdbarc']: # -> Teste si l'entrée après la virgule est une mot hébreu, si oui, l'ajoute à la liste
                entry = candidate
            else:
                break
    return words

def parse_row(rows):
    sort_key = 1
    bdb = []
    for row in rows:
        all_p = extract_definition(row['content'])
        for p in all_p:
            if p.find(['bdbheb', 'bdbarc']) is not None:
                d = p
                m = extract_headwords(d)
                if p.find('bdbarc'):
                    lang = "Aramaic"
                else:
                    lang = "Hebrew"
                bdb.append({
                    "sort_key": sort_key,
                    "strong": row['StrongNumber'] or None,
                    "lang": lang,
                    "m": m,
                    "b": [strip_vowels(w) for w in m],
                    "l": [t for w in m for t in (transliterate(w), transliterate(w, drop_gutturals=True))],
                    "d": str(d)
                    })
                sort_key += 1
    return bdb

In [ ]:
# ouvre les fichier csv 
bdb_rows = {}
csv.field_size_limit(sys.maxsize) # Allow to have the maxsize for each field

with open('bdb.csv', encoding='utf-8') as f:
    bdb_csv = csv.DictReader(f, delimiter='\t')
    rows = list(bdb_csv)

bdb = parse_row(rows)
#print(json.dumps(bdb[:20], ensure_ascii=False, indent=2))
# Enregistre le json
with open('bdb.json', 'w', encoding='utf-8') as f:
    json.dump(bdb, f, ensure_ascii=False, indent=2)

print("Fichier bdb.json créé")

Fichier BDB.json créé
